# 01. Данные и предобработка — вариант 4

Код восстановлен по PDF-отчёту. Результаты будут рассчитаны при выполнении; точное совпадение с отчётом не гарантируется. Запускайте ячейки сверху вниз.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src' / 'project.py').exists():
    raise RuntimeError('Запустите Jupyter из корня проекта, где находятся src/ и README.md')
sys.path.insert(0, str(ROOT / 'src'))
from project import *
frame, (X_train, X_test, y_train, y_test) = load_split()

In [ ]:
print('Размер:', frame.shape)
display(frame.dtypes.to_frame('dtype'))
display(frame.head())
display(pd.DataFrame({'missing': frame.isna().sum(), 'missing_share': frame.isna().mean()}))
print('Train:', X_train.shape, 'Test:', X_test.shape)
display(y_train.value_counts().to_frame('train_count'))
print('Меньшинство / большинство:', y_train.value_counts().min()/y_train.value_counts().max())

EDA выполняется только на train. duration исключён: при прогнозе до звонка этот признак неизвестен. unknown сохраняется как категория; pdays=-1 означает отсутствие предыдущих контактов.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
y_train.value_counts().sort_index().plot.bar(ax=axes[0], title='Классы train: 0=no, 1=yes')
X_train['age'].plot.hist(bins=30, ax=axes[1], title='Возраст')
plt.tight_layout(); plt.show()
X_train[NUM].hist(bins=30, figsize=(12, 8)); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
X_train['job'].value_counts().plot.barh(ax=axes[0], title='Профессии')
eda = X_train.assign(target=y_train)
eda.groupby('poutcome')['target'].mean().plot.bar(ax=axes[1], title='Доля депозитов по прошлому результату')
plt.tight_layout(); plt.show()

Числовые пропуски заполняются медианой, категориальные модой. OneHotEncoder обрабатывает неизвестные категории. Масштабирование используется только перед логистической регрессией. При дисбалансе применяются веса классов; SMOTE в этом проекте не используется.

In [ ]:
prep = preprocessing()
train_encoded = prep.fit_transform(X_train)
test_encoded = prep.transform(X_test)
print('После кодирования:', train_encoded.shape, test_encoded.shape)
print(prep.get_feature_names_out())